# Algorithmic Hiring Use Case, Rule Derivation for AI Governance
## Using `RuleDeriver` to derive appropriate compliance controls from `AttributeConditionRules`

This notebook demonstrates the `ai_atlas_nexus.blocks.rule_derivation.RuleDeriver` block using the same algorithmic hiring use case from [`hiring_usecase_comparison.ipynb`](hiring_usecase_comparison.ipynb).

The NY and EU hiring systems are similar: they share the same five risks, purpose, stakeholders, and capabilities. The only structural difference is `isUsedWithinLocality`.

The data includes `AttributeConditionRules` that fire when a system is used in a specific locality and derive which regulatory controls apply. The `RuleDeriver` block evaluates these rules and surfaces the derived attributes explicitly.

```
AI System (locality: NY)  ──▶  RuleDeriver  ──▶  derived_isUsedWithinLocality: ny
                                                   ──▶  NYC Local Law 144 audit required

AI System (locality: EU)  ──▶  RuleDeriver  ──▶  derived_isUsedWithinLocality: eu
                                                   ──▶  GDPR Art. 22 explanation required
```

## Setup

In [18]:
from pathlib import Path

from ai_atlas_nexus import AIAtlasNexus
from ai_atlas_nexus.blocks.atlas_explorer import OxigraphExplorer
from ai_atlas_nexus.blocks.rule_derivation import RuleDeriver, RuleDerivationResult, RuleMatch

SAMPLE_DATA_DIR = Path("sample_data")

print("Loading AI Atlas Nexus ontology...")
nexus = AIAtlasNexus(base_dir=str(SAMPLE_DATA_DIR))
explorer = OxigraphExplorer(nexus._ontology)

# Retrieve the two AI system defs
hiring_ny = explorer.get_by_id(None, "hiring-usecase-ny")
hiring_eu = explorer.get_by_id(None, "hiring-usecase-eu")

print(f"\nNY system  : {hiring_ny.name}")
print(f"  locality : {hiring_ny.isUsedWithinLocality}")
print(f"\nEU system  : {hiring_eu.name}")
print(f"  locality : {hiring_eu.isUsedWithinLocality}")

Loading AI Atlas Nexus ontology...


[2026-04-30 19:40:34:2] - INFO - AIAtlasNexus - Created AIAtlasNexus instance. Base_dir: sample_data



NY system  : Algorithmic Hiring (New York, US)
  locality : ['hiring-usecase-locality-ny-ny-usa']

EU system  : Algorithmic Hiring (EU)
  locality : ['hiring-usecase-locality-dublin-ie-eu']


## Section 1: The AttributeConditionRules

The hiring use case data contains two `AttributeConditionRules` —
one for each jurisdiction. Each rule has:

- **`preconditions`** — a slot condition that must be satisfied (e.g. `isUsedWithinLocality = NY`)
- **`postconditions`** — what to derive when the rule fires (propagate the locality to related risks)

These rules follow the `AttributeConditionRule` class defined in the AIAtlasNexus
`common.yaml` schema, which extends LinkML's `dpv:Rule`.

In [19]:
# Rules are stored in the ontology container alongside entries
rules = nexus._ontology.rules or []
attr_condition_rules = [r for r in rules if getattr(r, "type", None) == "AttributeConditionRule"]

print(f"Total rules in graph : {len(rules)}")
print(f"Total AttributeConditionRules : {len(attr_condition_rules)}")

print(f"Details on Rules from the two usecases")
for r in hiring_ny.hasRule + hiring_eu.hasRule:
    rule = explorer.get_by_id(None, r)
    print(f"\n{'─' * 60}")
    print(f"ID          : {rule.id}")
    print(f"Name        : {rule.name}")

    pre = rule.preconditions
    print(f"\nPreconditions (rule fires when ALL of these hold):")
    for sc in (pre.slot_conditions or []):
        print(f"  {sc.slot_name} == '{sc.equals_string}'")

    post = rule.postconditions
    print(f"\nPostconditions (derived when rule fires):")
    for sc in (post.slot_conditions or []):
        print(f"  {sc.slot_name} := '{sc.equals_string}'")

Total rules in graph : 182
Total AttributeConditionRules : 2
Details on Rules from the two usecases

────────────────────────────────────────────────────────────
ID          : hiring-usecase-locality-rule-ny-ny-usa
Name        : Hiring usecase rule, for New York, New York, USA

Preconditions (rule fires when ALL of these hold):
  isUsedWithinLocality == 'hiring-usecase-locality-ny-ny-usa'

Postconditions (derived when rule fires):
  hasRelatedRisk.isUsedWithinLocality := 'hiring-usecase-locality-ny-ny-usa'

────────────────────────────────────────────────────────────
ID          : hiring-usecase-locality-rule-dublin-ie-eu
Name        : Hiring usecase rule, for Dublin, Ireland, EU

Preconditions (rule fires when ALL of these hold):
  isUsedWithinLocality == 'hiring-usecase-locality-dublin-ie-eu'

Postconditions (derived when rule fires):
  hasRelatedRisk.isUsedWithinLocality := 'hiring-usecase-locality-dublin-ie-eu'


## Section 2: Initialise the RuleDeriver

`RuleDeriver` accepts the raw `AttributeConditionRule` list, either as Pydantic
objects or as raw dicts.

Internally it translates each rule into a `Predicate`/`Action` pair and builds a
priority-ordered `RulesEngine`. All postcondition paths that traverse a list
(e.g. `hasRelatedRisk.isUsedWithinLocality`) are simplified to a flat
`derived_<attr>` attribute on the root instance.

In [20]:
deriver = RuleDeriver([explorer.get_by_id(None, r) for r in hiring_ny.hasRule + hiring_eu.hasRule] or [])

print(f"RuleDeriver initialised")
print(f"  Translated rules    : {len(deriver.rules)}")
print(f"  Predicate slots     : {deriver.predicate_slots}")
print()

print("Translated rule priorities (highest evaluated first):")
for r in deriver.rules:
    print(f"  [{r.priority:>3}] {r.id}")
    print(f"         predicates : {[(p.path, p.op.value, p.value) for p in r.predicates]}")
    print(f"         actions    : {[(a.attribute, a.value) for a in r.actions]}")

RuleDeriver initialised
  Translated rules    : 2
  Predicate slots     : {'isUsedWithinLocality'}

Translated rule priorities (highest evaluated first):
  [  2] hiring-usecase-locality-rule-ny-ny-usa
         predicates : [('isUsedWithinLocality', 'eq', 'hiring-usecase-locality-ny-ny-usa')]
         actions    : [('derived_isUsedWithinLocality', 'hiring-usecase-locality-ny-ny-usa')]
  [  1] hiring-usecase-locality-rule-dublin-ie-eu
         predicates : [('isUsedWithinLocality', 'eq', 'hiring-usecase-locality-dublin-ie-eu')]
         actions    : [('derived_isUsedWithinLocality', 'hiring-usecase-locality-dublin-ie-eu')]


## Section 3: Explain — Dry-run Rule Evaluation

`deriver.explain(instance)` runs all rules against the instance **without mutating
anything** and returns a list of `RuleMatch` objects — one per rule — showing which
rules would fire and which derived attributes they would set.

This is useful for understanding and debugging the logic before applying it.

In [21]:
systems = [hiring_ny, hiring_eu]

for system in systems:
    print(f"\n{'=' * 60}")
    print(f"System: {system.name} ")
    print(f"Locality input: {system.isUsedWithinLocality}")

    # Pass the Pydantic object as a dict so _to_namespace can process it cleanly
    matches: list[RuleMatch] = deriver.explain(system.model_dump())

    print(f"\n{'─' * 40}")
    print("Rule evaluation (dry-run):")
    for m in matches:
        status = "✓ MATCHED" if m.matched else "  skipped"
        print(f"  {status}  {m.rule_id}  (priority {m.priority})")
        if m.matched:
            for attr, val in m.derived_attributes.items():
                print(f"               would set: {attr} = '{val}'")


System: Algorithmic Hiring (New York, US) 
Locality input: ['hiring-usecase-locality-ny-ny-usa']

────────────────────────────────────────
Rule evaluation (dry-run):
  ✓ MATCHED  hiring-usecase-locality-rule-ny-ny-usa  (priority 2)
               would set: derived_isUsedWithinLocality = 'hiring-usecase-locality-ny-ny-usa'
    skipped  hiring-usecase-locality-rule-dublin-ie-eu  (priority 1)

System: Algorithmic Hiring (EU) 
Locality input: ['hiring-usecase-locality-dublin-ie-eu']

────────────────────────────────────────
Rule evaluation (dry-run):
    skipped  hiring-usecase-locality-rule-ny-ny-usa  (priority 2)
  ✓ MATCHED  hiring-usecase-locality-rule-dublin-ie-eu  (priority 1)
               would set: derived_isUsedWithinLocality = 'hiring-usecase-locality-dublin-ie-eu'


## Section 4: Derive — Apply Rules and Collect Results

`deriver.derive(instance)` applies the rules and returns a `RuleDerivationResult`
containing:

- **`matches`** — per-rule match outcomes (same as `explain()`)
- **`derived`** — merged dict of all attributes set by fired rules
- **`fired`** — convenience property: only the matched `RuleMatch` objects

In [28]:
results: dict[str, RuleDerivationResult] = {}

for system in systems:
    result = deriver.derive(system.model_dump())
    print(result)
    results[system.id] = result

    print(f"\n{'=' * 60}")
    print(f"System       : {system.name}")
    print(f"Instance ID  : {result.instance_id}")
    print(f"\nFired rules  : {[m.rule_id for m in result.fired]}")
    print(f"Skipped rules: {[m.rule_id for m in result.matches if not m.matched]}")
    print(f"\nDerived attributes:")
    for attr, val in result.derived.items():
        print(f"  {attr} = '{val}'")
    if not result.derived:
        print("  (none)")

RuleDerivationResult(instance_id='hiring-usecase-ny', matches=[RuleMatch(rule_id='hiring-usecase-locality-rule-ny-ny-usa', description='Hiring usecase rule, for New York, New York, USA', priority=2, matched=True, derived_attributes={'derived_isUsedWithinLocality': 'hiring-usecase-locality-ny-ny-usa'}), RuleMatch(rule_id='hiring-usecase-locality-rule-dublin-ie-eu', description='Hiring usecase rule, for Dublin, Ireland, EU', priority=1, matched=False, derived_attributes={})], derived={'derived_isUsedWithinLocality': 'hiring-usecase-locality-ny-ny-usa'})

System       : Algorithmic Hiring (New York, US)
Instance ID  : hiring-usecase-ny

Fired rules  : ['hiring-usecase-locality-rule-ny-ny-usa']
Skipped rules: ['hiring-usecase-locality-rule-dublin-ie-eu']

Derived attributes:
  derived_isUsedWithinLocality = 'hiring-usecase-locality-ny-ny-usa'
RuleDerivationResult(instance_id='hiring-usecase-eu', matches=[RuleMatch(rule_id='hiring-usecase-locality-rule-ny-ny-usa', description='Hiring usecas

## Section 5: From Derived Attributes to Applicable Controls

The derived locality ID is now used to look up which controls are applicable
in that locality. Controls in the hiring data carry an `isApplicableinLocality`
field pointing to the locality for which they are required.

This step is domain-specific — the rule derivation block produces the derived
locality; the control lookup is a separate query into the ontology.

In [25]:
import yaml

# Load the raw YAML to access controls with their isApplicableinLocality field
hiring_data = yaml.safe_load((SAMPLE_DATA_DIR / "hiring_usecase_data.yaml").read_text())
controls = hiring_data.get("controls", [])

def find_controls_for_locality(locality_id: str, controls: list) -> list:
    return [c for c in controls if locality_id in c.get("isApplicableinLocality", [])]


for system in systems:
    result = results[system.id]

    print(f"\n{'=' * 60}")
    print(f"System: {system.name}")

    derived_locality = result.derived.get("derived_isUsedWithinLocality")
    if not derived_locality:
        print("  No locality derived — no locality-specific controls apply.")
        continue

    print(f"Derived locality : {derived_locality}")
    applicable = find_controls_for_locality(derived_locality, controls)

    if applicable:
        print(f"\nApplicable controls ({len(applicable)}):")
        for ctrl in applicable:
            print(f"\n  • {ctrl['name']}  [{ctrl['id']}]")
            desc = ctrl.get("description", "").strip().replace("\n", " ")
            print(f"    {desc}")
    else:
        print("  No controls found for this locality.")


System: Algorithmic Hiring (New York, US)
Derived locality : hiring-usecase-locality-ny-ny-usa

Applicable controls (1):

  • AuditAI requirements (NYC Local Law 144)  [hiring-control-audit-ai-ny]
    NYC Local Law 144 of 2021 requires annual independent bias audits of automated employment decision tools. This control addresses risks by ensuring periodic external assessment of discriminatory impact.

System: Algorithmic Hiring (EU)
Derived locality : hiring-usecase-locality-dublin-ie-eu

Applicable controls (1):

  • Provide explanation (GDPR Art. 22)  [hiring-control-gdpr-explanation-eu]
    GDPR Article 22 guarantees individuals the right to receive an explanation for automated hiring decisions that produce legal or similarly significant effects. This control addresses risks by ensuring transparency and accountability.


## Section 6: Summary Comparison

In [26]:
import pandas as pd

rows = []
for system in systems:
    result = results[system.id]
    derived_locality = result.derived.get("derived_isUsedWithinLocality", "—")
    applicable = find_controls_for_locality(derived_locality, controls)
    fired_rule = result.fired[0].rule_id if result.fired else "—"

    rows.append({
        "System": system.name,
        "Locality (input)": system.isUsedWithinLocality[0] if system.isUsedWithinLocality else "—",
        "Rule fired": fired_rule,
        "Derived locality": derived_locality,
        "Applicable controls": ", ".join(c["name"] for c in applicable) or "—",
    })

df = pd.DataFrame(rows).set_index("System")
pd.set_option("display.max_colwidth", 60)
df

,Locality (input),Rule fired,Derived locality,Applicable controls
System,,,,
"Algorithmic Hiring (New York, US)",hiring-usecase-locality-ny-ny-usa,hiring-usecase-locality-rule-ny-ny-usa,hiring-usecase-locality-ny-ny-usa,AuditAI requirements (NYC Local Law 144)
Algorithmic Hiring (EU),hiring-usecase-locality-dublin-ie-eu,hiring-usecase-locality-rule-dublin-ie-eu,hiring-usecase-locality-dublin-ie-eu,Provide explanation (GDPR Art. 22)


## Section 7: Connection to the Star Graph Analysis

The [`hiring_usecase_comparison.ipynb`](hiring_usecase_comparison.ipynb) star graph
comparison found a **94.4% Jaccard similarity** between the NY and EU systems, with
the only structural difference being the `isUsedWithinLocality` field.

In this notebook we showed the effect of a rule about the change of locality on the applicable risk controls.

| Layer | What it shows |
|---|---|
| Star graph comparison | *What* changed — a single locality field differs |
| Rule derivation | *Why* it matters — locality triggers different compliance rules |

The `AttributeConditionRules` encode the governance
knowledge that connects a system's deployment context(locality etc.) to its regulatory obligations.


### Certification validity

```
NY system certified under NYC Local Law 144
    │
    └─▶  deployed in EU
              │
              ├─▶  Star graph: 94.4% similar — looks safe
              │
              └─▶  Rule derivation: GDPR Art. 22 now applies, NYC Law 144 does NOT
                       ──▶  Original certification must be re-evaluated
```

In [27]:
# Demonstrate the rule derivation on a hypothetical cross-jurisdiction deployment:
# What if we applied the EU system's rules to the NY system's instance?

print("Scenario: NY system instance evaluated against EU rules")
print("(simulates asking 'what would change if we deployed this in the EU?')")
print()

result_ny = results["hiring-usecase-ny"]
result_eu = results["hiring-usecase-eu"]

ny_locality = result_ny.derived.get("derived_isUsedWithinLocality")
eu_locality = result_eu.derived.get("derived_isUsedWithinLocality")

ny_controls = find_controls_for_locality(ny_locality, controls) if ny_locality else []
eu_controls = find_controls_for_locality(eu_locality, controls) if eu_locality else []

ny_control_ids = {c["id"] for c in ny_controls}
eu_control_ids = {c["id"] for c in eu_controls}

controls_added   = eu_control_ids - ny_control_ids
controls_removed = ny_control_ids - eu_control_ids
controls_shared  = ny_control_ids & eu_control_ids

print(f"Controls shared (still required)  : {controls_shared or '(none)'}")
print(f"Controls REMOVED (no longer apply): {controls_removed or '(none)'}")
print(f"Controls ADDED (newly required)   : {controls_added or '(none)'}")
print()
print("Conclusion: the compliance obligations are entirely different.")
print("Re-certification is required before the EU deployment.")

Scenario: NY system instance evaluated against EU rules
(simulates asking 'what would change if we deployed this in the EU?')

Controls shared (still required)  : (none)
Controls REMOVED (no longer apply): {'hiring-control-audit-ai-ny'}
Controls ADDED (newly required)   : {'hiring-control-gdpr-explanation-eu'}

Conclusion: the compliance obligations are entirely different.
Re-certification is required before the EU deployment.


## Conclusion

The `RuleDeriver` block can derive governance attributes from `AttributeConditionRules` embedded in the ontology.

Key takeaways:

1. **`explain(instance)`** performs a dry-run — useful for auditing which rules
   apply to a given system without changing anything.

2. **`derive(instance)`** applies the rules and returns a `RuleDerivationResult`
   with typed fields (`fired`, `derived`, `matches`) for downstream processing.

3. Rules accept both **Pydantic objects** (from the loaded ontology) and **raw
   dicts** (from `yaml.safe_load`), making them easy to use in notebooks and scripts.

4. Combined with the star graph comparison from `hiring_usecase_comparison.ipynb`,
   rule derivation turns a structural similarity score into an **actionable
   compliance signal** — surfacing exactly which obligations change when a system
   crosses a jurisdictional boundary.